# 「A などの B」のパターンで上位語と下位語の組を集める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_lexical_semantics/hypernym_patterns.ipynb)

NLP クックブック 18-02 ／ 語の意味と意味の変化

「犬は動物の一種」のような上位・下位の関係（is-a 関係）を、テキストから自動的に集めたい。

In [1]:
!pip install -q datasets fugashi unidic-lite

In [2]:
import itertools, re
from collections import Counter, defaultdict
from datasets import load_dataset
from fugashi import Tagger

# 日本語 Wikipedia の先頭から 1.2 万記事の冒頭部分を使う (ストリーミングで必要な分だけ読む)
ds = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)
articles = [ex["text"][:1500] for ex in itertools.islice(ds, 12000)]
sentences = [s for a in articles for s in re.split(r"(?<=。)|\n", a) if 10 <= len(s) <= 150]
print(len(articles), "記事", len(sentences), "文")

12000 記事 322627 文


In [3]:
tagger = Tagger()

def chunks(sentence):
    """連続する名詞を 1 つの名詞句にまとめ、(文字列, 名詞句かどうか) の列にする"""
    out = []
    for w in tagger(sentence):
        is_noun = w.feature.pos1 in ("名詞", "接尾辞") and w.feature.pos2 != "数詞" or w.feature.pos1 == "接頭辞"
        if is_noun and out and out[-1][1]:
            out[-1] = (out[-1][0] + w.surface, True)
        else:
            out.append((w.surface, is_noun))
    return out

# パターン: 「A などの B」「A や C などの B」「A といった B」「A をはじめとする B」→ A (と C) は B の下位語
MARKERS = [("など", "の"), ("と", "いっ", "た"), ("を", "はじめ", "と", "する"), ("と", "いう")]
pairs = Counter()
for s in sentences:
    c = chunks(s)
    for i, (text, is_noun) in enumerate(c):
        if not is_noun:
            continue
        for marker in MARKERS:
            j = i + 1 + len(marker)
            if tuple(x for x, _ in c[i + 1:j]) == marker and j < len(c) and c[j][1]:
                hyponyms = [text]
                k = i
                while k >= 2 and c[k - 1][0] in ("や", "、", "・") and c[k - 2][1]:       # 「A や C などの B」の並列をさかのぼる
                    hyponyms.append(c[k - 2][0]); k -= 2
                for h in hyponyms:
                    pairs[h, c[j][0], "".join(marker)] += 1
print(len(pairs), "組")
print(Counter(m for _, _, m in pairs).most_common())

13277 組
[('などの', 8301), ('という', 3105), ('といった', 1646), ('をはじめとする', 225)]


In [4]:
# 上位語ごとに下位語をまとめる
by_hypernym = defaultdict(Counter)
for (hypo, hyper, marker), n in pairs.items():
    if marker != "という" and len(hypo) >= 2 and len(hyper) >= 2:
        by_hypernym[hyper][hypo] += n
for hyper in ["言語", "スポーツ", "動物", "施設", "楽器", "金属", "作品", "問題"]:
    print(hyper, "←", " ".join(w for w, _ in by_hypernym[hyper].most_common(15)))

print("\n下位語の種類が多い上位語:", [(h, len(c)) for h, c in sorted(by_hypernym.items(), key=lambda x: -len(x[1]))[:15]])
print("「という」の例:", [(a, b) for (a, b, m), n in pairs.most_common(3000) if m == "という"][:12])

言語 ← Tcl ウイグル語 モンゴル語 上海語 日本語 Prolog Scheme LISP Lua アセンブリ言語 Ruby Python Perl ウゴル語派 フィン
スポーツ ← 野球 スキー サッカー 陸上競技 乗馬 登山 傍ら ソフトボール オリエンテーリング 海水浴 テニス スノーボード 一方
動物 ← ヒト マウス サル ヘビ カエル 爬虫類 海獣類 両生類 無脊椎動物 魚介類 恐竜 木の葉 ウサギ ヤギ
施設 ← 線路 空港ターミナルビル 停車場 列車 育成場 城壁 装備 柏戸病院 千葉大学附属病院 千葉県文化会館 千葉市立郷土博物館 千葉県立中央図書館 ZOZOマリンスタジアム 駐車場 野球場
楽器 ← シンクラヴィア フェアライトCMI 大型オルガン 歴史的ピアノ
金属 ← フリーデル振動 ウィグナー結晶化 プラズモン 遮蔽 ステンレス やすい鉄 ニッケル
作品 ← 安部公房 村上春樹 三島由紀夫 大友克洋 吾妻ひでお 山岸凉子 大島弓子 萩尾望都 手塚治虫 永島慎二 岡田史子 花田清輝 渡辺一夫 小林秀雄 石川淳
問題 ← 貧困 汚職 テロ 医療 不定性 露光不足 手ぶれ 過密状態 低減 水漏れ オーバーヒート 互換性 アルコール依存症 負重性蹄葉炎 非識字

下位語の種類が多い上位語: [('分野', 83), ('多く', 68), ('影響', 49), ('研究', 43), ('機能', 41), ('総称', 38), ('一部', 38), ('こと', 37), ('作品', 37), ('場合', 36), ('要素', 35), ('概念', 34), ('施設', 34), ('意味', 33), ('種類', 32)]
「という」の例: [('偶数', '条件'), ('ニオブ', '名前'), ('列車', '表現'), ('コロンビウム', '名前'), ('国体', '語'), ('ため', '説'), ('死刑', '特定'), ('者', '意味'), ('四神', '形'), ('にくい', '特徴'), ('漫画家', '職業'), ('年', '長期間')]


- 「A などの B」のような言い回しは、A が B の一種であることを高い確率で示す。英語の "B such as A" にあたり、Hearst パターンと呼ばれる。
- パターンごとに精度が違う。「などの」「といった」は信頼でき、「という」は「〜という意味」「〜という名前」のような上位語でない組を多く拾う。パターン別に抜き取って精度を確かめる。
- 「問題」「もの」「作品」のような抽象的な語は、何の上位語にもなれてしまう。下位語の種類が極端に多い上位語は、辞書として使う前に除くか、別扱いにする。
- 並列（「A、B などの C」）をさかのぼるとき、文の区切りを越えて無関係な語（「一方」「傍ら」）を拾うことがある。係り受け解析を使えば減らせる。
- 名詞句の範囲の取り方（どこまでを 1 語とするか）で結果が大きく変わる。ここでは連続する名詞をまとめただけなので、「の」でつながる長い名詞句は切れている。
- 再現率は低い（パターンに当てはまる文にしか効かない）。大量のテキストに適用して数で補うか、次項のブートストラッピングでパターンを増やす。